# 03 · Nested spatial cross-validation (recomputed, same as V3)

```
for each outer fold k = 1..5 (V3's blockCV folds):
    test  = development boreholes in fold k
    train = the other boreholes (with a purge: minus those closer than the purge distance)
    inner CV = the other 4 outer folds (V3's backup inner folds if one is too small or one-sided)
        Optuna tries hyperparameter sets (50 for RF/XGB/MLP, 30 for GWRF/GWXGB) and the
        weight of filled cells; each is scored by the inner Macro-F1 at its best threshold
    refit the best set on all training boreholes -> predict fold k
```

| Block size | Purge | Models |
|---|---|---|
| 106 m (main) | none | RF, XGB, MLP, GWRF, GWXGB |
| 212 m and 318 m | none | RF, XGB |
| 106 m | 106 m and 212 m | RF, XGB |

**Step 1, pilot (5 minutes):** with `PILOT = True` (the default) only XGB, fold 1, is run and compared with V3's saved predictions. It must say **identical**.
**Step 2, full run:** set `PILOT = False` and run again. It saves after every fold; if it stops, run again and it continues.

**Run time of the full run:** about the same as V3's (roughly 7-15 hours on the school PC).

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
from tqdm import tqdm          # text progress bars

PILOT = True           # True: XGB fold 1 only, compared with V3.  False: everything.
RUN_ONLY = None        # e.g. ["_p0_"] for the runs without purge first

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
runs = H.run_list(S)
outer = S["quick"]["outer_folds"] if S["run"]["quick"] else list(range(1, S["blockcv"]["k"] + 1))
if PILOT:
    runs, outer = [r for r in runs if r["model"] == "XGB" and r["purge_m"] == 0][:1], [1]
elif RUN_ONLY:
    runs = [r for r in runs if any(s in r["tag"] for s in RUN_ONLY)]
PRIV = H.out_dir(S, "03_spatial_cv", private=True)
(PRIV / "checkpoints").mkdir(exist_ok=True); (PRIV / "models").mkdir(exist_ok=True)
main_block = H.block_sizes(S)[0]
display(pd.DataFrame(runs))

## Run (or continue)
One bar per run; it moves one step per Optuna trial. Next to it: the outer folds finished so far, with the best and mean Macro-F1 and accuracy on their test boreholes.

In [ ]:
def fold_score(pred):
    return H.macro_f1(pred.y, pred.y_pred), float((pred.y == pred.y_pred).mean())

for r in runs:
    per_fold = H.n_trials(r["model"], S)
    done = {k: fold_score(joblib.load(PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib")["predictions"])
            for k in outer if (PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib").exists()}
    bar = tqdm(total=len(outer) * per_fold, initial=len(done) * per_fold, desc=f"{r['tag']:<20}",
               bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} trials [{elapsed}<{remaining}]{postfix}")

    def show():
        if done:
            f1, acc = [v[0] for v in done.values()], [v[1] for v in done.values()]
            best = max(done, key=lambda k: done[k][0])
            bar.set_postfix_str(f"folds {len(done)}/{len(outer)} | best fold {best}: F1 {done[best][0]:.3f} "
                                f"acc {done[best][1]:.3f} | mean F1 {np.mean(f1):.3f} acc {np.mean(acc):.3f}")
    show()
    for k in outer:
        if k in done:
            continue
        res = H.run_fold(r, dev, k, S, on_trial=lambda: bar.update(1))
        if r["block_m"] == main_block and r["purge_m"] == 0:     # fold models of the main runs, for notebook 06
            joblib.dump(res["model"], PRIV / "models" / f"{r['tag']}_fold{k}.joblib", compress=3)
        res.pop("model")
        joblib.dump(res, PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib", compress=3)
        done[k] = fold_score(res["predictions"])
        show()
    bar.close()
print(f"Done ({(time.time() - T0) / 3600:.2f} h in this session).")

## Pilot check against V3 (only with PILOT = True)

In [ ]:
if PILOT:
    r = runs[0]
    mine = joblib.load(PRIV / "checkpoints" / f"{r['tag']}_fold1.joblib")["predictions"]
    mine.to_csv(PRIV / "pilot.csv", index=False)
    mine = pd.read_csv(PRIV / "pilot.csv")      # saved and read back like V3's file (a CSV can change the last digit)
    v3 = pd.read_csv(H.v3_outputs(S, "03_nested_cv", "restricted") / "outer_predictions.csv")
    v3 = v3[(v3.model == r["model"]) & (v3.block_m == r["block_m"]) & (v3.purge_m == 0) & (v3.outer_fold == 1)]
    same = len(mine) == len(v3) and np.array_equal(mine.p_clay.values, v3.p_clay.values) \
        and np.array_equal(mine.y_pred.values, v3.y_pred.values)
    print(f"Pilot {r['tag']} fold 1: {len(mine)} test cells | IDENTICAL to V3: {same}")
    if same:
        print("Good: set PILOT = False and run the notebook again for the full run.")
    else:
        print("NOT identical: do not start the full run; send this notebook to Claude.")
        print("largest difference in P(Clay):", np.abs(mine.p_clay.values - v3.p_clay.values).max() if len(mine) == len(v3) else "different cells")

## Collect the results (after the full run)

In [ ]:
if not PILOT:
    preds, infos = [], []
    for f in sorted((PRIV / "checkpoints").glob("*.joblib")):
        res = joblib.load(f)
        preds.append(res["predictions"]); infos.append(res["info"])
    preds = pd.concat(preds, ignore_index=True)
    preds.to_csv(PRIV / "outer_predictions.csv", index=False)
    info = pd.DataFrame(infos)
    info.drop(columns=["params"]).to_csv(H.out_dir(S, "03_spatial_cv") / "run_info.csv", index=False)
    display(info.groupby("tag").agg(folds=("outer_fold", "count"), hours=("seconds", lambda s: s.sum() / 3600),
                                    inner_folds=("inner_folds", lambda s: ", ".join(sorted(set(s))))).round(2))
    print(f"{len(preds)} test predictions saved. Next: notebook 04.")